# Organized execution copy

The exact executed predecessor with historical outputs is in `archive/revision_executed/`. This execution copy has cleared outputs. No new GPU experiment was run during packaging. Install, restart, then run the configuration cell and remaining cells in order. Edit DATA_ROOT and RUN_DIR consistently across notebooks; use a new RUN_DIR for a new reproduction and the same one to resume it. See `docs/RUN_ORDER.md` and `docs/DATA_SETUP.md` for required files. The conversion/model/scoring calls and experimental settings are retained.


In [ ]:
# Cell 1 — INSTALL
# After completion: Runtime → Restart session

import os
import sys
import subprocess

print("Python:", sys.version, flush=True)

OPENVOICE_REPO = "/content/OpenVoice"
OPENVOICE_COMMIT = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

repositories = [
    (
        OPENVOICE_REPO,
        "https://github.com/myshell-ai/OpenVoice.git",
        OPENVOICE_COMMIT,
    ),
    (
        SEED_REPO,
        "https://github.com/Plachtaa/seed-vc.git",
        SEED_COMMIT,
    ),
]

for repo, url, commit in repositories:
    print(f"\nPreparing {repo}...", flush=True)

    if not os.path.isdir(repo):
        subprocess.run(
            ["git", "clone", "--progress", url, repo],
            check=True,
        )

    subprocess.run(
        ["git", "checkout", commit],
        cwd=repo,
        check=True,
    )

    print(f"Repository ready: {repo}", flush=True)

pkgs = [
    # OpenVoice
    "wavmark==0.0.3",
    "eng_to_ipa==0.0.2",
    "inflect==7.0.0",
    "unidecode==1.3.7",
    "pypinyin==0.50.0",
    "cn2an==0.5.22",
    "jieba==0.42.1",
    "langid==1.1.6",

    # Seed-VC
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml",

    # Evaluation
    "speechbrain==1.0.3",
    "jiwer==3.0.3",
    "accelerate",
]

print("\nInstalling packages...", flush=True)

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
    ] + pkgs,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Package installation failed: {return_code}"
    )

print(
    "\n✅ Installed. NOW: Runtime → Restart session.",
    flush=True,
)

In [ ]:
# Runtime configuration — edit these two lines consistently in all notebooks.
import os
DATA_ROOT = "/content/drive/MyDrive/IJEEE_VC_data"
RUN_DIR = "/content/drive/MyDrive/IJEEE_VC_data/reproduction_runs/review_run_01"
# DATA_ROOT contains audio/models. RUN_DIR is a new output folder, never results/.
os.environ["IJEEE_DATA_ROOT"] = DATA_ROOT
os.environ["IJEEE_RUN_DIR"] = RUN_DIR


In [ ]:
# Cell 2 — PATHS AND SETTINGS

import os
import sys
import glob
import json
import time
import random
import shutil
import tempfile
import re
import string
import unicodedata
import subprocess

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

from google.colab import drive

drive.mount("/content/drive")

import numpy as np
import pandas as pd
import torch

BASE_DIR = os.environ.get("IJEEE_DATA_ROOT", "/content/drive/MyDrive/IJEEE_VC_data")

OUT = os.environ.get("IJEEE_RUN_DIR", os.path.join(BASE_DIR, "reproduction_runs", "review_run_01"))
os.makedirs(OUT, exist_ok=True)

TUNING_FILES = [
    os.path.join(
        BASE_DIR,
        "selected_hybrid_data",
        f"joe_source_{i:02d}.wav",
    )
    for i in (1, 2, 3)
]

TUNING_TEXT = [
    "The monkey was planning to throw the net like the fishermen did.",
    "Gaze out to sea, picnic on benches and watch fishermen hook herring.",
    "We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money.",
]

TARGET_REF = os.path.join(
    BASE_DIR,
    "selected_hybrid_data",
    "kathleen_reference.wav",
)

EXTRA_REFS = [
    os.path.join(
        BASE_DIR,
        "selected_hybrid_data",
        f"kathleen_conditioning_ref_{i:02d}.wav",
    )
    for i in (2, 3)
]

EVAL_CLIPS = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "kathleen_independent_refs",
            "kathleen_eval_*.wav",
        )
    )
)

JOE_ARCHIVE = os.path.join(BASE_DIR, "joe.tar.gz")

HOLDOUT10_META = os.path.join(
    BASE_DIR,
    "final_holdout_data",
    "holdout_metadata.csv",
)

HOLDOUT40_DIR = os.path.join(
    BASE_DIR,
    "final_holdout_data_40",
)
os.makedirs(HOLDOUT40_DIR, exist_ok=True)

HOLDOUT40_META = os.path.join(
    HOLDOUT40_DIR,
    "holdout40_metadata.csv",
)

EXISTING = os.path.join(
    BASE_DIR,
    "final_hybrid_comparison",
)

OV_CONVERTER = os.path.join(
    BASE_DIR,
    "models",
    "OpenVoiceV2",
    "converter",
)

N_HOLDOUT = 40

TUNING_IDS = {
    "0000000160",
    "0000000229",
    "0000000068",
}

required_files = TUNING_FILES + [
    TARGET_REF,
    JOE_ARCHIVE,
    HOLDOUT10_META,
    os.path.join(OV_CONVERTER, "config.json"),
    os.path.join(OV_CONVERTER, "checkpoint.pth"),
]

for path in required_files:
    assert os.path.isfile(path), f"❌ missing: {path}"

for path in EXTRA_REFS:
    print(
        "extra reference",
        os.path.basename(path),
        "found" if os.path.isfile(path) else "❌ NOT FOUND",
    )

print("Independent Kathleen evaluation clips:", len(EVAL_CLIPS))

assert len(EVAL_CLIPS) == 10, (
    "❌ kathleen_eval_*.wav clips not found"
)

assert torch.cuda.is_available(), (
    "❌ Select a T4 GPU runtime"
)

print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def append_row(csv_path, row):
    pd.DataFrame([row]).to_csv(
        csv_path,
        mode="a",
        header=not os.path.exists(csv_path),
        index=False,
    )


def done_keys(csv_path, columns):
    if not os.path.exists(csv_path):
        return set()

    table = pd.read_csv(csv_path)

    return set(
        map(
            tuple,
            table[columns].astype(str).values,
        )
    )


print("✅ Paths ready")

In [ ]:
# Cell 3 — PREPARE THE SAME 40 HOLDOUT SENTENCES

from pathlib import Path
import tarfile

LOCAL_JOE = "/content/final_comparison_dataset/joe"
os.makedirs(LOCAL_JOE, exist_ok=True)

if not glob.glob(
    os.path.join(LOCAL_JOE, "**", "*.webm"),
    recursive=True,
):
    print("Extracting Joe recordings...", flush=True)

    with tarfile.open(JOE_ARCHIVE, "r:gz") as archive:
        archive.extractall(LOCAL_JOE, filter="data")

audio_files = []

for extension in [
    "*.webm",
    "*.wav",
    "*.flac",
    "*.mp3",
    "*.ogg",
    "*.opus",
    "*.m4a",
]:
    audio_files.extend(
        glob.glob(
            os.path.join(LOCAL_JOE, "**", extension),
            recursive=True,
        )
    )

audio_files = sorted(set(audio_files))
print("Joe files:", len(audio_files))

candidates = [
    path
    for path in audio_files
    if Path(path).stem not in TUNING_IDS
]

random.Random(20260822).shuffle(candidates)


def dur(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def txt(path):
    transcript_path = Path(path).with_suffix(".txt")

    if not transcript_path.is_file():
        return None

    transcript = " ".join(
        transcript_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).split()
    )

    return transcript or None


records = []

for path in candidates:
    if len(records) == N_HOLDOUT:
        break

    duration = dur(path)

    if duration is None or not (4.0 < duration < 7.5):
        continue

    transcript = txt(path)

    if transcript is None:
        continue

    sample = len(records) + 1

    output_path = os.path.join(
        HOLDOUT40_DIR,
        f"joe_holdout_{sample:02d}.wav",
    )

    if not os.path.isfile(output_path):
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v", "error",
                "-i", path,
                "-ac", "1",
                "-ar", "22050",
                "-c:a", "pcm_s16le",
                output_path,
            ],
            check=True,
        )

    records.append(
        {
            "sample": sample,
            "original_id": Path(path).stem,
            "duration_seconds": duration,
            "transcript": transcript,
            "prepared_wav": output_path,
        }
    )

    print(
        f"Prepared {sample}/{N_HOLDOUT}: {Path(path).stem}",
        flush=True,
    )

assert len(records) == N_HOLDOUT, (
    f"❌ Expected {N_HOLDOUT} sentences, found {len(records)}"
)

H40 = pd.DataFrame(records)

old10 = pd.read_csv(
    HOLDOUT10_META,
    dtype={"original_id": str},
)

same = (
    H40["original_id"].astype(str).head(10).tolist()
    == old10["original_id"].astype(str).tolist()
)

print("First 10 identical to the paper's holdout:", same)

assert same, (
    "❌ Selection differs from the original holdout — stop and report"
)

# Verify all 40 against existing metadata before saving.
if os.path.isfile(HOLDOUT40_META):
    previous = pd.read_csv(
        HOLDOUT40_META,
        dtype={"original_id": str},
    )

    assert (
        H40["original_id"].tolist()
        == previous["original_id"].tolist()
    ), "❌ The 40 recording IDs differ from the saved experiment"

    assert (
        H40["transcript"].tolist()
        == previous["transcript"].tolist()
    ), "❌ The transcripts differ from the saved experiment"

H40.to_csv(HOLDOUT40_META, index=False)

print(
    H40[
        ["sample", "original_id", "duration_seconds"]
    ].tail(5)
)

print("✅ 40 holdout sentences ready")

In [ ]:
# Cell 4 — LOAD MODELS, WARM-UP, CONVERSION FUNCTIONS

from types import SimpleNamespace
import importlib

sys.path.insert(0, "/content/OpenVoice")
importlib.invalidate_caches()

from openvoice.api import ToneColorConverter

print("Loading OpenVoice...", flush=True)

tone_color_converter = ToneColorConverter(
    os.path.join(OV_CONVERTER, "config.json"),
    device="cuda:0",
)

tone_color_converter.load_ckpt(
    os.path.join(OV_CONVERTER, "checkpoint.pth")
)

sys.path.insert(0, "/content/seed-vc")
os.chdir("/content/seed-vc")

import inference as seed_inference


def seed_args(src, ref, out, steps):
    return SimpleNamespace(
        source=src,
        target=ref,
        output=out,
        diffusion_steps=steps,
        length_adjust=1.0,
        inference_cfg_rate=0.7,
        f0_condition=False,
        auto_f0_adjust=False,
        semi_tone_shift=0,
        checkpoint=None,
        config=None,
        fp16=True,
    )


print("Loading Seed-VC...", flush=True)

seed_model_bundle = seed_inference.load_models(
    seed_args(None, None, None, 10)
)

# Reuse loaded models during conversion.
seed_inference.load_models = lambda args: seed_model_bundle

SE_CACHE = {}


def get_se(path):
    if path not in SE_CACHE:
        SE_CACHE[path] = tone_color_converter.extract_se(path)

    return SE_CACHE[path]


def run_openvoice(src, ref, tau, out_path, seed):
    # Embedding extraction is outside this stage's timed span.
    src_se = get_se(src)
    tgt_se = get_se(ref)

    set_all_seeds(seed)

    torch.cuda.synchronize()
    start = time.perf_counter()

    tone_color_converter.convert(
        audio_src_path=src,
        src_se=src_se,
        tgt_se=tgt_se,
        output_path=out_path,
        tau=tau,
        message="@MyShell",
    )

    torch.cuda.synchronize()

    return time.perf_counter() - start


os.makedirs("/content/seed_tmp", exist_ok=True)


def run_seedvc(src, ref, steps, out_path, seed):
    work = tempfile.mkdtemp(dir="/content/seed_tmp")

    try:
        set_all_seeds(seed)

        torch.cuda.synchronize()
        start = time.perf_counter()

        seed_inference.main(
            seed_args(src, ref, work, steps)
        )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        produced = sorted(
            glob.glob(os.path.join(work, "*.wav")),
            key=os.path.getmtime,
        )

        assert produced, "❌ Seed-VC produced no file"

        shutil.move(produced[-1], out_path)

        return elapsed

    finally:
        shutil.rmtree(work, ignore_errors=True)


def run_cascade_e2e(
    src,
    ref,
    tau,
    steps,
    out_path,
    ov_seed,
    sv_seed,
):
    intermediate = out_path.replace(
        ".wav",
        "_e2e_ov.wav",
    )

    torch.cuda.synchronize()
    start = time.perf_counter()

    run_openvoice(
        src,
        ref,
        tau,
        intermediate,
        ov_seed,
    )

    run_seedvc(
        intermediate,
        ref,
        steps,
        out_path,
        sv_seed,
    )

    torch.cuda.synchronize()

    return time.perf_counter() - start


WARM = "/content/warmup"
os.makedirs(WARM, exist_ok=True)

for repetition in range(1, 3):
    print(f"Warm-up {repetition}/2...", flush=True)

    run_openvoice(
        TUNING_FILES[0],
        TARGET_REF,
        0.5,
        os.path.join(WARM, "ov.wav"),
        2026,
    )

    run_seedvc(
        TUNING_FILES[0],
        TARGET_REF,
        10,
        os.path.join(WARM, "sv.wav"),
        2028,
    )

ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "openvoice_commit": (
        "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"
    ),
    "seedvc_commit": (
        "51383efd921027683c89e5348211d93ff12ac2a8"
    ),
    "seedvc_fp16": True,
    "openvoice_precision": "fp32",
    "batch_size": 1,
    "warmups": 2,
    "timed_repeats": 3,
    "cuda_synchronize": True,
    "openvoice_embeddings_cached": True,
}

# Keep a separate environment record for this new session.
environment_path = os.path.join(
    OUT,
    f"environment_extra_{time.strftime('%Y%m%d_%H%M%S')}.json",
)

with open(environment_path, "w", encoding="utf-8") as file:
    json.dump(ENV, file, indent=2)

print("✅ Models loaded and warmed up")
print(ENV)

In [ ]:
# Cell 5 — SCORING MODELS AND FUNCTIONS

import librosa
import jiwer
import torchaudio
import logging
import torch.nn.functional as F

if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]

from speechbrain.inference.speaker import SpeakerRecognition

from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    AutoFeatureExtractor,
    WavLMForXVector,
)

logging.getLogger("speechbrain").setLevel(logging.WARNING)

DEV = "cuda"

print("Loading ECAPA...", flush=True)

speaker_model = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/evaluation_models/ecapa_voxceleb",
    run_opts={"device": DEV},
)

ASR = {}

for name, model_id, dtype in [
    ("small", "openai/whisper-small", torch.float32),
    ("large", "openai/whisper-large-v3", torch.float16),
]:
    print(f"Loading Whisper-{name}...", flush=True)

    processor = AutoProcessor.from_pretrained(model_id)

    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id,
        torch_dtype=dtype,
        use_safetensors=True,
    ).to(DEV).eval()

    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None

    ASR[name] = (processor, model, dtype)

print("Loading WavLM...", flush=True)

wl_fe = AutoFeatureExtractor.from_pretrained(
    "microsoft/wavlm-base-plus-sv"
)

wl = WavLMForXVector.from_pretrained(
    "microsoft/wavlm-base-plus-sv"
).to(DEV).eval()

print("Loading UTMOS...", flush=True)

utmos = torch.hub.load(
    "tarepan/SpeechMOS:v1.2.0",
    "utmos22_strong",
    trust_repo=True,
).to(DEV).eval()


def load_audio_16k(path):
    audio, _ = librosa.load(
        path,
        sr=16000,
        mono=True,
    )
    return np.asarray(audio, dtype=np.float32)


_EMB = {}


@torch.inference_mode()
def emb(path, kind):
    key = (path, kind)

    if key in _EMB:
        return _EMB[key]

    audio = load_audio_16k(path)

    if kind == "ecapa":
        embedding = speaker_model.encode_batch(
            torch.from_numpy(audio).unsqueeze(0).to(DEV),
            wav_lens=torch.tensor([1.0]).to(DEV),
            normalize=False,
        )

        embedding = F.normalize(
            embedding.squeeze().float(),
            p=2,
            dim=0,
        ).cpu()

    else:
        inputs = wl_fe(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
        ).to(DEV)

        embedding = F.normalize(
            wl(**inputs).embeddings.squeeze().float(),
            p=2,
            dim=0,
        ).cpu()

    _EMB[key] = embedding
    return embedding


def cos(first, second, kind):
    return float(
        F.cosine_similarity(
            emb(first, kind),
            emb(second, kind),
            dim=0,
        )
    )


def join_spelled_letters(text):
    # Apply the same letter-joining rule used previously.
    return re.sub(
        r"\b(?:[A-Z] )+[A-Z]\b",
        lambda match: match.group(0).replace(" ", ""),
        str(text),
    )


def normalize_transcript(text):
    text = unicodedata.normalize(
        "NFKC",
        join_spelled_letters(text),
    ).lower()

    for old, new in [
        ("’", "'"),
        ("–", " "),
        ("—", " "),
        ("-", " "),
    ]:
        text = text.replace(old, new)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    return re.sub(r"\s+", " ", text).strip()


@torch.inference_mode()
def transcribe(path, which):
    processor, model, dtype = ASR[which]

    inputs = processor(
        load_audio_16k(path),
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True,
    )

    arguments = {
        "input_features": inputs.input_features.to(DEV, dtype),
        "language": "en",
        "task": "transcribe",
        "max_new_tokens": 128,
    }

    if getattr(inputs, "attention_mask", None) is not None:
        arguments["attention_mask"] = inputs.attention_mask.to(DEV)

    generated = model.generate(**arguments)

    return processor.batch_decode(
        generated,
        skip_special_tokens=True,
    )[0].strip()


@torch.inference_mode()
def mos(path):
    audio = torch.from_numpy(
        load_audio_16k(path)
    ).unsqueeze(0).to(DEV)

    return float(utmos(audio, 16000).item())


def score(output, text, source, conditioning_reference, evaluation_clips):
    results = {
        "sim_ecapa_cond": cos(
            output, conditioning_reference, "ecapa"
        ),
        "sim_wavlm_cond": cos(
            output, conditioning_reference, "wavlm"
        ),
        "sim_ecapa_eval": float(
            np.mean([
                cos(output, clip, "ecapa")
                for clip in evaluation_clips
            ])
        ),
        "sim_wavlm_eval": float(
            np.mean([
                cos(output, clip, "wavlm")
                for clip in evaluation_clips
            ])
        ),
        "leak_ecapa_source": cos(
            output, source, "ecapa"
        ),
        "utmos": mos(output),
        "duration_out": librosa.get_duration(path=output),
    }

    reference_normalized = normalize_transcript(text)

    for which in ("small", "large"):
        hypothesis = transcribe(output, which)
        hypothesis_normalized = normalize_transcript(hypothesis)

        results[f"hyp_{which}"] = hypothesis

        results[f"wer_{which}"] = jiwer.wer(
            reference_normalized,
            hypothesis_normalized,
        )

        results[f"cer_{which}"] = jiwer.cer(
            reference_normalized,
            hypothesis_normalized,
        )

        results["ref_words"] = len(reference_normalized.split())

    # Remove the output embedding while retaining reference embeddings.
    for key in list(_EMB):
        if key[0] == output:
            _EMB.pop(key)

    return results


print("✅ Scoring models ready")

In [ ]:
# Cell 6 — REPRODUCTION CHECK

H40 = pd.read_csv(
    HOLDOUT40_META,
    dtype={"original_id": str},
)

existing_audio = os.path.join(
    EXISTING,
    "seed_based_outputs",
    "optimized_hybrid",
    "optimized_hybrid_sample_01_tau_0.5_steps_10.wav",
)

assert os.path.isfile(existing_audio), (
    f"❌ Missing file: {existing_audio}"
)

sample = H40.iloc[0]

existing_scores = score(
    existing_audio,
    sample["transcript"],
    sample["prepared_wav"],
    TARGET_REF,
    EVAL_CLIPS,
)

print(
    f"Existing output: "
    f"similarity {existing_scores['sim_ecapa_cond']:.4f} "
    f"(paper CSV 0.6885) | "
    f"WER small {existing_scores['wer_small']:.4f} "
    f"(paper 0.1667)",
    flush=True,
)

assert abs(
    existing_scores["sim_ecapa_cond"] - 0.6885
) < 0.01, "❌ Scoring differs from the paper — stop and report"

check_audio = os.path.join(
    OUT,
    "_check_selected_s01.wav",
)

intermediate_audio = check_audio.replace(
    ".wav",
    "_ov.wav",
)

run_openvoice(
    sample["prepared_wav"],
    TARGET_REF,
    0.5,
    intermediate_audio,
    202701,
)

run_seedvc(
    intermediate_audio,
    TARGET_REF,
    10,
    check_audio,
    202901,
)

regenerated_scores = score(
    check_audio,
    sample["transcript"],
    sample["prepared_wav"],
    TARGET_REF,
    EVAL_CLIPS,
)

print(
    f"Regenerated with the paper's seeds: "
    f"similarity {regenerated_scores['sim_ecapa_cond']:.4f} | "
    f"transcript: {regenerated_scores['hyp_small']}",
    flush=True,
)

assert abs(
    regenerated_scores["sim_ecapa_cond"] - 0.6885
) < 0.03, "❌ Regenerated output differs — stop and report"

print("✅ REPRODUCTION CHECK PASSED")

In [ ]:
# EXTRA RUN — 40 sentences × 3 configurations

import uuid

# Keep this ID when rerunning the cell in the same session.
if "TAU_EXTRA_RUN_ID" not in globals():
    TAU_EXTRA_RUN_ID = (
        time.strftime("%Y%m%d_%H%M%S")
        + "_"
        + uuid.uuid4().hex[:8]
    )

prefix = f"E14_extra_{TAU_EXTRA_RUN_ID}"
RESULTS_PATH = os.path.join(OUT, prefix + ".csv")
AUDIO_DIR = os.path.join(OUT, prefix + "_audio")

os.makedirs(AUDIO_DIR, exist_ok=True)

holdout = pd.read_csv(
    HOLDOUT40_META,
    dtype={"original_id": str},
).sort_values("sample")

assert len(holdout) == 40
assert set(holdout["sample"]) == set(range(1, 41))

SYSTEMS_EXTRA = [
    (
        "OpenVoice-SeedVC (tau 0.1, 10 steps)",
        "ov_sv",
        0.1,
        10,
    ),
    (
        "SeedVC-OpenVoice (tau 0.3, 20 steps)",
        "sv_ov",
        0.3,
        20,
    ),
    (
        "SeedVC-OpenVoice (tau 0.5, 10 steps)",
        "sv_ov",
        0.5,
        10,
    ),
]

if os.path.isfile(RESULTS_PATH):
    saved = pd.read_csv(RESULTS_PATH)
    assert not saved.duplicated(["system", "sample"]).any()

    completed = {
        (row["system"], int(row["sample"]))
        for _, row in saved.iterrows()
    }
else:
    saved = pd.DataFrame()
    completed = set()

metadata_path = os.path.join(OUT, prefix + ".json")

if not os.path.exists(metadata_path):
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(
            {
                "run_id": TAU_EXTRA_RUN_ID,
                "gpu": torch.cuda.get_device_name(0),
                "environment": ENV,
                "holdout_metadata": HOLDOUT40_META,
                "conditioning_reference": TARGET_REF,
                "evaluation_clips": EVAL_CLIPS,
                "systems": SYSTEMS_EXTRA,
                "timing_repeats": 3,
                "end_to_end_repeats": 1,
                "openvoice_seed": "202700 + sample",
                "seedvc_seed": "202900 + sample",
                "results_path": RESULTS_PATH,
            },
            file,
            indent=2,
        )

print("Run ID:", TAU_EXTRA_RUN_ID, flush=True)
print("Results:", RESULTS_PATH, flush=True)
print("Already saved:", len(completed), "/120", flush=True)


def run_extra_pair(
    source,
    order,
    tau,
    steps,
    intermediate,
    output,
    ov_seed,
    sv_seed,
):
    if order == "ov_sv":
        ov_time = run_openvoice(
            source,
            TARGET_REF,
            tau,
            intermediate,
            ov_seed,
        )

        sv_time = run_seedvc(
            intermediate,
            TARGET_REF,
            steps,
            output,
            sv_seed,
        )
    else:
        sv_time = run_seedvc(
            source,
            TARGET_REF,
            steps,
            intermediate,
            sv_seed,
        )

        ov_time = run_openvoice(
            intermediate,
            TARGET_REF,
            tau,
            output,
            ov_seed,
        )

    return ov_time, sv_time


for name, order, tau, steps in SYSTEMS_EXTRA:
    tag = re.sub(r"[^A-Za-z0-9]+", "_", name)

    for _, record in holdout.iterrows():
        sample = int(record["sample"])
        key = (name, sample)

        if key in completed:
            continue

        source = record["prepared_wav"]
        duration = float(record["duration_seconds"])

        output = os.path.join(
            AUDIO_DIR,
            f"{tag}_s{sample:02d}.wav",
        )

        intermediate = os.path.join(
            AUDIO_DIR,
            f"{tag}_s{sample:02d}_intermediate.wav",
        )

        ov_seed = 202700 + sample
        sv_seed = 202900 + sample

        print(
            f"\n{name} | Sample {sample}/40",
            flush=True,
        )

        ov_times = []
        sv_times = []

        # Same seeds across the three timing repetitions.
        for repetition in range(1, 4):
            ov_time, sv_time = run_extra_pair(
                source,
                order,
                tau,
                steps,
                intermediate,
                output,
                ov_seed,
                sv_seed,
            )

            ov_times.append(ov_time)
            sv_times.append(sv_time)

            print(
                f"Repeat {repetition}/3 | "
                f"OpenVoice={ov_time:.3f}s | "
                f"Seed-VC={sv_time:.3f}s",
                flush=True,
            )

        # Separate end-to-end run, including both stages.
        e2e_output = output.replace(".wav", "_e2e.wav")
        e2e_intermediate = intermediate.replace(
            ".wav", "_e2e.wav"
        )

        torch.cuda.synchronize()
        start = time.perf_counter()

        run_extra_pair(
            source,
            order,
            tau,
            steps,
            e2e_intermediate,
            e2e_output,
            ov_seed,
            sv_seed,
        )

        torch.cuda.synchronize()
        latency_e2e = time.perf_counter() - start

        mean_ov = float(np.mean(ov_times))
        mean_sv = float(np.mean(sv_times))
        latency = mean_ov + mean_sv

        row = {
            "run_id": TAU_EXTRA_RUN_ID,
            "system": name,
            "sample": sample,
            "order": order,
            "tau": tau,
            "steps": steps,
            "ov_seed": ov_seed,
            "sv_seed": sv_seed,
            "src_duration": duration,
            "t_openvoice_run1": ov_times[0],
            "t_openvoice_run2": ov_times[1],
            "t_openvoice_run3": ov_times[2],
            "t_seedvc_run1": sv_times[0],
            "t_seedvc_run2": sv_times[1],
            "t_seedvc_run3": sv_times[2],
            "t_openvoice": mean_ov,
            "t_seedvc": mean_sv,
            "latency": latency,
            "latency_e2e": latency_e2e,
            "rtf": latency / duration,
        }

        print("Scoring...", flush=True)

        row.update(
            score(
                output,
                record["transcript"],
                source,
                TARGET_REF,
                EVAL_CLIPS,
            )
        )

        frame = pd.DataFrame([row])

        if not saved.empty:
            assert set(frame.columns) == set(saved.columns), (
                "❌ Results columns differ — stop and report"
            )
            frame = frame.reindex(columns=saved.columns)

        updated = pd.concat(
            [saved, frame],
            ignore_index=True,
        )

        # Write a complete temporary CSV before replacing the result file.
        temporary_path = RESULTS_PATH + ".tmp"
        updated.to_csv(temporary_path, index=False)
        os.replace(temporary_path, RESULTS_PATH)

        saved = updated
        completed.add(key)

        print(
            f"✅ Saved {len(completed)}/120",
            flush=True,
        )

results = pd.read_csv(RESULTS_PATH)

assert not results.duplicated(["system", "sample"]).any()

for name, _, _, _ in SYSTEMS_EXTRA:
    samples = results.loc[
        results["system"] == name,
        "sample",
    ]
    assert len(samples) == 40
    assert set(samples) == set(range(1, 41))

assert len(results) == 120

print("\nResults per system:")
print(results.groupby("system").size())

print(
    results.groupby("system")[
        [
            "sim_ecapa_cond",
            "sim_ecapa_eval",
            "sim_wavlm_eval",
            "wer_large",
            "cer_large",
            "utmos",
            "latency",
            "latency_e2e",
            "rtf",
        ]
    ].mean().round(4)
)

print("\n✅ Extra run completed — 120 results")
print("Results file:", RESULTS_PATH)

In [ ]:
# Cell 13 — PACK RESULTS AND DOWNLOAD

import os
import glob
import zipfile

from google.colab import files

assert "RESULTS_PATH" in globals(), "Run EXTRA RUN first in this session, or explicitly set RESULTS_PATH to its CSV."
extra_csv = RESULTS_PATH

extra_json = extra_csv.replace(".csv", ".json")

assert os.path.isfile(extra_csv), (
    f"❌ Missing extra-run results: {extra_csv}"
)

assert os.path.isfile(extra_json), (
    f"❌ Missing extra-run settings: {extra_json}"
)

extra = pd.read_csv(extra_csv)

assert len(extra) == 120
assert not extra.duplicated(["system", "sample"]).any()

for system, group in extra.groupby("system"):
    assert len(group) == 40
    assert set(group["sample"]) == set(range(1, 41))

assert extra["system"].nunique() == 3

result_files = sorted(
    set(
        glob.glob(os.path.join(OUT, "*.csv"))
        + glob.glob(os.path.join(OUT, "*.json"))
        + [HOLDOUT40_META]
    )
)

for path in result_files:
    assert os.path.isfile(path), f"❌ Missing file: {path}"

archive_names = [
    os.path.basename(path)
    for path in result_files
]

assert len(archive_names) == len(set(archive_names)), (
    "❌ Duplicate filenames in the ZIP"
)

zip_path = "/content/IJEEE_revision_results_with_extra_run.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in result_files:
        archive.write(
            path,
            arcname=os.path.basename(path),
        )

with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None, "❌ ZIP verification failed"

    print("Files included:")
    for name in archive.namelist():
        print(" -", name)

print("\n✅ Results ZIP ready")
print("✅ Extra run verified: 120 results")

files.download(zip_path)